SECCIONES 5 Y 6

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Separar características (X) y variable objetivo (y)
X = df.drop(columns=['SalePrice', 'SalePrice_log'])

y_log = df['SalePrice_log']

# 2. Separación entrenamiento/prueba (Sección 6)
# Usamos un split de 80% entrenamiento y 20% prueba con semilla para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

print(f"Tamaño del set de entrenamiento: {X_train.shape}")
print(f"Tamaño del set de prueba: {X_test.shape}")

SECCION 7

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# 1. Identificación de tipos de variables
nan_is_none_cols = [
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
    'GarageQual', 'GarageCond', 'MasVnrType'
]

# Variables de calidad (Ordinales)
quality_cols = ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual']
quality_mapping = ['Po', 'Fa', 'TA', 'Gd', 'Ex']

# Separar variables numéricas y categóricas nominales restantes
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_nominal_cols = [col for col in X_train.select_dtypes(include=['object']).columns
                    if col not in nan_is_none_cols and col not in quality_cols]

# 2. Definición de Transformadores (Pipelines)

# A. Faltantes intencionales (NaN = 'None') + OneHotEncoding
none_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# B. Variables numéricas (Faltantes reales -> Mediana)
num_transformer = SimpleImputer(strategy='median')

# C. Variables categóricas ordinales (Faltantes -> Moda + OrdinalEncoding)
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[quality_mapping] * len(quality_cols)))
])

# D. Variables categóricas nominales (Faltantes -> Moda + OneHotEncoding)
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Ensamblar el preprocesador (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('nan_none', none_transformer, nan_is_none_cols),
        ('num', num_transformer, num_cols),
        ('ordinal', ordinal_transformer, quality_cols),
        ('nominal', nominal_transformer, cat_nominal_cols)
    ],
    remainder='passthrough'
)

# Ajustar transformaciones solo con datos de entrenamiento y aplicarlas a Test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

SECCION 8

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

# 1. Instanciar y entrenar el modelo base
dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(X_train_processed, y_train)

# 2. Realizar predicciones
y_pred_train_dummy = dummy_regr.predict(X_train_processed)
y_pred_test_dummy = dummy_regr.predict(X_test_processed)

# 3. Evaluar el modelo (Root Mean Squared Error)
rmse_train_dummy = np.sqrt(mean_squared_error(y_train, y_pred_train_dummy))
rmse_test_dummy = np.sqrt(mean_squared_error(y_test, y_pred_test_dummy))

print("--- Evaluación Modelo Base (DummyRegressor) ---")
print(f"RMSE (Train): {rmse_train_dummy:.4f}")
print(f"RMSE (Test):  {rmse_test_dummy:.4f}")